<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_3_Generation_and_Limits/23_3_2_What_LLMs_Get_Wrong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 7
## What Language Models Get Wrong

---

## What This Notebook Is About

GPT-2 in 23_3_1 continues whatever text it's given; it doesn't answer questions. The chatbots most people use are language models of the same kind that have had two more stages of training. First they are fine-tuned on many examples of an instruction followed by a good response (**instruction tuning**), the same idea as fine-tuning DistilBERT on reviews in 23_2_2. Usually they are then trained further to give the kinds of answers people rate highly. The result answers questions in a helpful-sounding way, but underneath it is still choosing likely next tokens, one at a time.

This notebook uses a small model of that kind, **SmolLM2-360M-Instruct** (Hugging Face, 2024), with 362 million weights, to look at the ways these models go wrong. The chatbots you use are much larger and make these mistakes less often, but they make every kind of mistake shown here, and the reasons are the same.

**What you will learn:**
1. How a chat model is given a conversation, and what it gets right
2. **Hallucination**: fluent, confident answers that are made up, and what the model's own probabilities say about them
3. Mistakes that come from working with tokens instead of letters
4. How the form of a question changes the answer, and why that makes models hard to evaluate
5. How to tell whether a model has memorized a text, and why that matters for test sets

The model runs on the CPU in this notebook (a 720 MB download). It takes about a minute on Colab's CPU to answer the questions in Section 4, and the other cells take seconds. Every answer is chosen greedily, the most likely token each time, so the answers you get should match the saved ones.

In [1]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cpu'
name = 'HuggingFaceTB/SmolLM2-360M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
print(f'{name}: {sum(p.numel() for p in model.parameters()):,} weights')

def ask(question, max_tokens=60, show_probs=False):
    # Format the question as a chat, generate an answer greedily, and return the answer's text.
    chat = tokenizer.apply_chat_template([{'role': 'user', 'content': question}], add_generation_prompt=True,
                                         return_tensors='pt', return_dict=True).to(device)
    with torch.no_grad():
        out = model.generate(**chat, max_new_tokens=max_tokens, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)
    new_ids = out.sequences[0, chat['input_ids'].shape[1]:]
    if show_probs:
        # the probability the model gave each token of its answer when it chose it
        probs = [scores[0].softmax(dim=0)[t].item() for scores, t in zip(out.scores, new_ids)]
        print('  '.join(f'{tokenizer.decode(int(t))!r} {p:.2f}' for t, p in zip(new_ids, probs)
                        if t != tokenizer.eos_token_id))
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-360M-Instruct: 361,821,120 weights


`ask` sends one question to the model and returns its answer. The question first goes through the model's **chat template**, which wraps it in the special tokens the model saw during instruction tuning, marking where the user's message starts and ends and where the assistant's answer begins. Here is what the model actually receives:

In [2]:
print(tokenizer.apply_chat_template([{'role': 'user', 'content': 'What is the capital of Australia?'}],
                                    add_generation_prompt=True, tokenize=False))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of Australia?<|im_end|>
<|im_start|>assistant



The model sees more than the question. The template adds a **system message** that comes before every conversation ("You are a helpful AI assistant named SmolLM, trained by Hugging Face"), marks each message with `<|im_start|>` and `<|im_end|>` tokens and the speaker's role, and ends with the start of an assistant message, so that the most likely next tokens are an answer. The model was fine-tuned on conversations in exactly this format. Chatbots work the same way, usually with a much longer system message that the user never sees.

---

## Section 1: What It Gets Right

In [3]:
for question in ['What is the capital of Australia? Answer in a few words.',
                 'Who wrote the novel Pride and Prejudice? Answer in a few words.',
                 'In what year did the first person walk on the Moon? Answer in a few words.',
                 'What is 17 times 23?']:
    print(f'Q: {question}\nA: {ask(question)}\n')

Q: What is the capital of Australia? Answer in a few words.
A: The capital of Australia is Canberra.



Q: Who wrote the novel Pride and Prejudice? Answer in a few words.
A: Jane Austen



Q: In what year did the first person walk on the Moon? Answer in a few words.
A: 1969



Q: What is 17 times 23?
A: 17 times 23 is 391.



All four are right: Canberra, Jane Austen, 1969, and 17 × 23 = 391. These facts appear many times in the text the model was trained on, and for a question like this, the most likely continuation is the right answer.

---

## Section 2: Making Things Up

There is no Nobel Prize in Basket Weaving, and no 2011 novel called *The Glass Lantern of Mirewood*; both were invented for this notebook.

In [4]:
for question in ['Who won the 1987 Nobel Prize in Basket Weaving?',
                 'Summarize the plot of the 2011 novel "The Glass Lantern of Mirewood" by Helena Castell in two sentences.']:
    print(f'Q: {question}\nA: {ask(question)}\n')

Q: Who won the 1987 Nobel Prize in Basket Weaving?
A: The 1987 Nobel Prize in Basket Weaving was awarded to the Swedish engineer and inventor, Gunnar Björnsson, for his invention of the "Basket Weaving Machine."



Q: Summarize the plot of the 2011 novel "The Glass Lantern of Mirewood" by Helena Castell in two sentences.
A: In "The Glass Lantern of Mirewood," Helena Castell's novel, a young woman named Aria discovers a mysterious glass lantern that grants her the power to heal and heal others. As she delves deeper into the lantern's secrets, she must navigate a world of magic and politics, where



Both answers are fluent and specific, and both are invented. The model names a winner, "the Swedish engineer and inventor, Gunnar Björnsson", and a reason, a "Basket Weaving Machine". For the novel, it gives a heroine called Aria and a magic lantern, and runs out of tokens mid-sentence. Nothing in the answers suggests any doubt.

This is called **hallucination**. It follows from how the model works: it produces the tokens most likely to follow the question, and the most likely continuation of "Who won the 1987 Nobel Prize in..." is the name of a winner, because that's how such questions are answered in its training text. Whether the prize exists isn't part of that calculation. 23_2_1 showed the same thing at a smaller scale, when DistilBERT's top guess for the capital of France was Marseille. Instruction tuning teaches a model the form of a good answer, and only partly teaches it to say "I don't know".

The answer reads as confidently as the right ones, but the model's own probabilities tell a different story. `show_probs=True` prints each token of the answer with the probability the model gave it when it was chosen.

In [5]:
print('A real fact:')
ask('What is the capital of Australia? Answer in a few words.', show_probs=True)
print('\nAn invented one:')
ask('Who won the 1987 Nobel Prize in Basket Weaving?', max_tokens=28, show_probs=True)

A real fact:


'The' 0.28  ' capital' 1.00  ' of' 0.99  ' Australia' 1.00  ' is' 1.00  ' Canberra' 0.99  '.' 0.92

An invented one:


'The' 0.23  ' ' 0.70  '1' 1.00  '9' 1.00  '8' 1.00  '7' 1.00  ' Nobel' 1.00  ' Prize' 0.99  ' in' 1.00  ' Basket' 0.78  ' We' 0.75  'aving' 1.00  ' was' 0.91  ' awarded' 0.92  ' to' 0.99  ' the' 0.08  ' Swedish' 0.15  ' engineer' 0.18  ' and' 0.29  ' inventor' 0.45  ',' 0.27  ' Gun' 0.09  'nar' 0.99  ' Bj' 0.17  'ör' 0.56  'n' 0.41  'sson' 0.72  ',' 0.62


'The 1987 Nobel Prize in Basket Weaving was awarded to the Swedish engineer and inventor, Gunnar Björnsson,'

For the real fact, every token after the first is chosen with a probability of 0.92 or more, and " Canberra" with 0.99: the model was sure of every step. For the invented answer, the first part is just as sure, because it repeats the question ("1987 Nobel Prize in Basket Weaving"). Then the probabilities drop: " the" 0.08, " Swedish" 0.15, " engineer" 0.18, " Gun" 0.09, " Bj" 0.17. At each of those points, many other tokens were nearly as likely, and greedy decoding picked one. A different but similar question might have produced a Norwegian chemist.

So the uncertainty is there inside the model, even though the text doesn't show it. Some research systems use token probabilities like these to flag answers that are likely to be made up. It isn't a complete test, because a model can also be confidently wrong about something it saw wrongly many times in training.

---

## Section 3: Tokens, Not Letters

In [6]:
print('A:', ask('How many times does the letter r appear in the word strawberry?'))
print('tokens:', tokenizer.tokenize(' strawberry'), tokenizer.tokenize('strawberry'))

A: The word "strawberry" contains 10 occurrences of the letter "r".
tokens: ['Ġstrawberry'] ['st', 'raw', 'berry']


There are 3 r's in "strawberry", and the model says 10. With a space in front, as it appears in the middle of a sentence, "strawberry" is a single token, `Ġstrawberry` (the `Ġ` marks the space, as in 23_0_1's GPT-2 tokenizer). At the start of a text it is three tokens, `st`, `raw` and `berry`. Either way, the model never sees the letters: it gets one or three IDs, each with an embedding (23_0_2). To count the r's it would have to have learned how each of those tokens is spelled, which it was never directly taught. Larger models do better at this, but questions about the letters in a word are a well-known weak spot of token-based models.

---

## Section 4: Same Question, Different Form

Ten easy general-knowledge questions, each with four options, asked three ways:

1. As a multiple-choice question, answered with a letter, with the options in one order
2. The same, with the options in reverse order
3. As an open question, answered in a few words, with no options

For the open answers we also need a rule for deciding whether an answer is right. Two common rules are an **exact match** with the reference answer, after removing a final period and ignoring capital letters, and **containing** the reference answer anywhere in the reply.

In [7]:
questions = [
    ('Which planet is closest to the Sun?', ['Venus', 'Mercury', 'Mars', 'Earth'], 'Mercury'),
    ('Which gas do plants take in from the air to make sugar?', ['Oxygen', 'Nitrogen', 'Carbon dioxide', 'Helium'], 'Carbon dioxide'),
    ('What is the largest ocean on Earth?', ['Atlantic', 'Indian', 'Arctic', 'Pacific'], 'Pacific'),
    ('Who painted the Mona Lisa?', ['Michelangelo', 'Leonardo da Vinci', 'Raphael', 'Donatello'], 'Leonardo da Vinci'),
    ('How many legs does a spider have?', ['Six', 'Eight', 'Ten', 'Twelve'], 'Eight'),
    ('What is the boiling point of water at sea level in degrees Celsius?', ['90', '100', '110', '120'], '100'),
    ('Which element has the chemical symbol O?', ['Gold', 'Osmium', 'Oxygen', 'Iron'], 'Oxygen'),
    ('In which country are the pyramids of Giza?', ['Mexico', 'Peru', 'Egypt', 'Sudan'], 'Egypt'),
    ('What is the square root of 81?', ['7', '8', '9', '10'], '9'),
    ('Which animal is the largest mammal?', ['Elephant', 'Blue whale', 'Giraffe', 'Hippopotamus'], 'Blue whale'),
]
LETTERS = 'ABCD'

def multiple_choice(question, options):
    prompt = question + '\n' + '\n'.join(f'{L}) {o}' for L, o in zip(LETTERS, options)) + '\nAnswer with the letter only.'
    letter = ask(prompt, max_tokens=4)[:1]
    return letter, options[LETTERS.index(letter)] if letter in LETTERS else '?'

score = {'letters, original order': 0, 'letters, reversed order': 0, 'open, exact match': 0, 'open, contains': 0}
print(f"{'reference':18} {'original order':22} {'reversed order':22} open answer")
for question, options, reference in questions:
    letter1, choice1 = multiple_choice(question, options)
    letter2, choice2 = multiple_choice(question, options[::-1])
    open_answer = ask(question + ' Answer in a few words.', max_tokens=20)
    score['letters, original order'] += choice1 == reference
    score['letters, reversed order'] += choice2 == reference
    score['open, exact match'] += open_answer.rstrip('.').lower() == reference.lower()
    score['open, contains'] += reference.lower() in open_answer.lower()
    print(f'{reference:18} {letter1} {choice1:20} {letter2} {choice2:20} {open_answer}')
print()
for way, n in score.items():
    print(f'{way:26} {n} of {len(questions)}')

reference          original order         reversed order         open answer


Mercury            C Mars                 C Mercury              Mercury


Carbon dioxide     C Carbon dioxide       C Nitrogen             Plants take in carbon dioxide from the air to make sugar.


Pacific            D Pacific              D Atlantic             The largest ocean on Earth is the Pacific Ocean.


Leonardo da Vinci  C Raphael              C Leonardo da Vinci    Leonardo da Vinci.


Eight              D Twelve               A Twelve               A spider has 8 legs.


100                A 90                   A 120                  Boiling point of water at sea level is 100 degrees Celsius.


Oxygen             C Oxygen               D Gold                 Oxygen


Egypt              D Sudan                C Peru                 Giza


9                  C 9                    A 10                   The square root of 81 is 9.


Blue whale         D Hippopotamus         D Elephant             The largest mammal is the blue whale.

letters, original order    4 of 10
letters, reversed order    2 of 10
open, exact match          3 of 10
open, contains             8 of 10


The open answers are almost all right. Read by a person, 9 of the 10 are correct: "A spider has 8 legs" is right, and only "Giza" for the country of the pyramids misses the question. As multiple-choice questions answered with a letter, the same model gets only 4 of 10 right with the options in one order and 2 of 10 with them reversed. Its choice changed in 9 of the 10 questions when the order changed, and in six of them it gave the same letter both times, whatever option was in that position. Nine of its twenty letter answers were C, and seven were D. Asked for a letter, this small model largely answers by position rather than by content. Larger models are much better at multiple-choice questions, but they also show smaller versions of this order effect, and benchmarks now often shuffle the options for that reason.

The scoring rule matters as much. Exact match gives the open answers 3 of 10, because the model usually answers in a sentence ("The largest ocean on Earth is the Pacific Ocean."). The "contains" rule gives 8 of 10: it misses "A spider has 8 legs", because the reference is the word "Eight", and it correctly fails "Giza". So the same model, on the same ten questions, scores anywhere from 2 to 8 out of 10 by automatic rules and 9 by a person's reading, and none of these numbers is wrong. Each one answers a different question about the model.

---

## Section 5: Has It Seen This Before?

23_3_1 showed that the loss measures how expected a text is. A text the model saw many times during training, and partly memorized, should have a much lower loss than a text that says the same thing in different words. Here are the openings of two famous texts that appear in countless places on the web, each next to a version with the same meaning in different words.

In [8]:
def loss_of(text):
    ids = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    with torch.no_grad():
        return model(ids, labels=ids).loss.item()

pairs = [
    ('It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness',
     'It was the finest of days, it was the most dreadful of days, it was a time of good sense, it was a time of silliness'),
    ('We hold these truths to be self-evident, that all men are created equal, that they are endowed by their Creator with certain unalienable Rights',
     'We consider these facts to be obvious, that all people are made equal, that they are given by their maker certain rights that cannot be taken away'),
]
for original, reworded in pairs:
    print(f'loss {loss_of(original):.3f}   {original[:70]}...')
    print(f'loss {loss_of(reworded):.3f}   {reworded[:70]}...\n')

loss 0.945   It was the best of times, it was the worst of times, it was the age of...


loss 2.582   It was the finest of days, it was the most dreadful of days, it was a ...



loss 0.562   We hold these truths to be self-evident, that all men are created equa...


loss 3.143   We consider these facts to be obvious, that all people are made equal,...



The opening of *A Tale of Two Cities* has a loss of 0.945, against 2.582 for the reworded version. The opening of the US Declaration of Independence has a loss of 0.562, against 3.143 reworded. In 23_3_1, an ordinary sentence had a loss of about 3. These famous openings are far more predictable to the model than ordinary text, because it has seen them, word for word, many times. The reworded versions say the same thing but aren't in its training data, so the model has to predict them the way it predicts any new text.

This matters for evaluation. A model is often judged by its score on a published **benchmark**, a fixed set of test questions. If the questions and answers were on the web before the model was trained, the model may have seen them, and its score partly measures memory instead of ability. This is called **contamination**, and it's 19_1_1's rule about test sets broken by accident: the test examples played a part in training. Comparing the loss on the published questions with the loss on reworded versions, as above, is one of the ways researchers check for it.

---

## Section 6: Using These Models Carefully

None of the problems in this notebook is a bug that can be fixed in one place. They follow from what the model is: a network trained to produce likely text, then fine-tuned to produce the kind of answers people like. Some practical consequences:

- **Check facts against a source.** A fluent, confident answer is what the model produces whether or not it knows. Many current systems connect the model to a search engine or a set of documents and have it answer from what it retrieves (the embedding search from 23_0_2 is one way to do the retrieval), which reduces made-up answers but doesn't eliminate them.
- **Don't ask it to do what tokens hide.** Counting letters, reversing words and exact arithmetic on long numbers are harder for a model that sees tokens, not characters or digits. Larger models do better, and systems that let the model run code (a calculator or a short program) avoid the problem.
- **Measure on your own task, the way you'll use it.** Section 4 showed that the same ten questions can give scores anywhere from 2 to 8 out of 10, depending on the form of the question and the scoring rule, when a person reading the answers would give 9. That is 19_1_1's lesson about test sets, with one more step: a reported score means little unless you know exactly how it was measured, and the only score that tells you about your use is one measured on examples like yours, in the form you'll use.
- **Be suspicious of benchmarks the model may have seen.** If test questions were on the web before the model was trained, a high score may partly be memory, and Section 5 showed one way to check.

---

## Summary

- Chat models are language models fine-tuned on instructions and responses. They still produce one likely token at a time, and the **chat template** wraps each message in special tokens.
- The model answered real facts correctly, but it also gave a fluent, specific answer about a prize that doesn't exist and a novel that was never written (**hallucination**). Its own token probabilities were low exactly where it was making things up (0.08 to 0.18), even though the text sounded sure.
- It said "strawberry" has 10 r's. The model sees tokens, not letters, so questions about spelling are hard for it.
- The same ten questions scored 4 of 10 and 2 of 10 as multiple choice with the options in two orders, 3 and 8 of 10 as open questions under two scoring rules, and 9 of 10 by a person's reading. A score means little without knowing how it was measured.
- Famous texts have a much lower loss than the same ideas reworded (0.945 against 2.582, and 0.562 against 3.143), a sign the model memorized them. Test questions that were in the training data inflate benchmark scores (**contamination**).

## What's next

This is the last notebook of the course. The course started in 19_0 by using a pretrained image model as a black box, before you knew what a weight was. Since then you have opened up every part of it: the neuron and softmax (19_1), how networks learn from the loss by gradient descent (unit 20), the training loop and how to train well (unit 21), convolution, transfer learning and the ways image models fail (unit 22), and tokens, embeddings, attention, fine-tuning and generation (unit 23). The models you'll meet after this course are larger, but they are built from these parts, and the habits from 19_1 still apply to them: compare with a baseline, keep a test set that played no part in your choices, look at the mistakes, and find out what the training data was.